# Maritime Demo Deployment
This notebook deploys all Fabric items from the local Git repository to a Fabric workspace.

**Prerequisites:**
- Azure CLI installed and authenticated (`az login`)
- OR you can use interactive device code authentication

In [1]:
%pip install azure-identity requests fabric-cicd -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
import json
import requests
from pathlib import Path
from azure.identity import AzureCliCredential, DeviceCodeCredential, ChainedTokenCredential

# 1. Get the local repository path (current notebook's directory)
notebook_dir = Path(__file__).parent if '__file__' in globals() else Path.cwd()
repo_path = str(notebook_dir.absolute())

print(f"📂 Local repository path: {repo_path}")

# 2. Authenticate to Fabric - try Azure CLI first, fallback to device code
print("\n🔐 Authenticating to Microsoft Fabric...")
print("💡 TIP: If you get 'UserNotLicensed', you need a Fabric trial or license.")
print("   Visit: https://app.fabric.microsoft.com → Start trial (free 60 days)\n")

try:
    # Use DeviceCodeCredential first for explicit account selection
    credential = DeviceCodeCredential()
    fabric_token = credential.get_token("https://api.fabric.microsoft.com/.default").token
    print("✅ Authentication successful!")
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    sys.exit(1)

# Store credential for later use
globals()['credential'] = credential

# 3. List available workspaces
print("\n📋 Fetching your Fabric workspaces...")
headers = {
    "Authorization": f"Bearer {fabric_token}",
    "Content-Type": "application/json"
}

response = requests.get(
    "https://api.fabric.microsoft.com/v1/workspaces",
    headers=headers
)

if response.status_code == 200:
    workspaces = response.json().get("value", [])
    print(f"\nFound {len(workspaces)} workspace(s):")
    for idx, ws in enumerate(workspaces, 1):
        print(f"  {idx}. {ws['displayName']} (ID: {ws['id']})")
elif response.status_code == 401 and "UserNotLicensed" in response.text:
    print(f"❌ License Required: Your account doesn't have Fabric access.")
    print("\n🔧 Solutions:")
    print("  1. Start a FREE 60-day Fabric trial:")
    print("     → https://app.fabric.microsoft.com (click 'Start trial')")
    print("  2. Use a different Microsoft account with Fabric access")
    print("  3. Contact your admin to assign a Fabric capacity/license")
    sys.exit(1)
else:
    print(f"❌ Failed to list workspaces: {response.status_code} - {response.text}")
    sys.exit(1)

# Store for next cell
globals()['fabric_token'] = fabric_token
globals()['workspaces'] = workspaces
globals()['repo_path'] = repo_path

📂 Local repository path: /Users/rabindori/workspace/aitour2/maritimedemo/fabricdemo

🔐 Authenticating to Microsoft Fabric...
💡 TIP: If you get 'UserNotLicensed', you need a Fabric trial or license.
   Visit: https://app.fabric.microsoft.com → Start trial (free 60 days)

To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code LXMAL6ELB to authenticate.
✅ Authentication successful!

📋 Fetching your Fabric workspaces...

Found 43 workspace(s):
  1. My workspace (ID: cdb1026b-80d9-4c77-810f-1ad47a9134f9)
  2. ingest_fabric (ID: e76f9e48-86b2-4f53-8636-4dcb91cc66e2)
  3. ws-testDGW (ID: 5eccb735-d59f-401c-9aaf-9ea110ff21a4)
  4. ws-powerbi (ID: 84721c9d-29a6-4f0d-b8b5-3cffd12c1290)
  5. testpython (ID: 2eec9a69-93a5-47cb-a7ab-219937825ed5)
  6. test-vnetdgw (ID: 3fbc6b2e-dd79-4782-bdbe-e1816bc2fb90)
  7. VBD Fabric Real-time Analytics Tutorial (ID: 902a3a27-4035-46cf-a4dc-5c0d12bf20a4)
  8. Fabric Lakehouse Tutorial (ID: 300aa1d5-7a8c-4e1f-8c24-f

In [3]:
# Check who is currently authenticated
import base64

# Decode the JWT token to see user info
try:
    # JWT tokens have 3 parts separated by dots: header.payload.signature
    token_parts = fabric_token.split('.')
    if len(token_parts) >= 2:
        # Decode the payload (add padding if needed)
        payload = token_parts[1]
        # Add padding for base64 decoding
        payload += '=' * (4 - len(payload) % 4)
        decoded = base64.urlsafe_b64decode(payload)
        token_info = json.loads(decoded)
        
        print("🔍 Currently authenticated user:")
        print(f"   Email/UPN: {token_info.get('upn', token_info.get('unique_name', 'N/A'))}")
        print(f"   Name: {token_info.get('name', 'N/A')}")
        print(f"   App ID: {token_info.get('appid', 'N/A')}")
        
        if 'oid' in token_info:
            print(f"   Object ID: {token_info['oid']}")
            
        print("\n💡 If this is not the correct account, run 'az logout' then 'az login' with the right account")
    else:
        print("⚠️ Could not decode token")
except Exception as e:
    print(f"⚠️ Could not identify user from token: {e}")
    print("\nTo check Azure CLI account, run: az account show")

🔍 Currently authenticated user:
   Email/UPN: admin@MngEnvMCAP534753.onmicrosoft.com
   Name: System Administrator
   App ID: 04b07795-8ddb-461a-bbee-02f9e1bf7b46
   Object ID: c8b2a1e9-0268-4042-b884-70cfd42be415

💡 If this is not the correct account, run 'az logout' then 'az login' with the right account


In [4]:
# 4. Select target workspace
# Option A: Use the first workspace (or specify index)
workspace_index = 42  # Change this to select a different workspace

if workspace_index >= len(workspaces):
    print(f"❌ Invalid workspace index. You have {len(workspaces)} workspace(s).")
    sys.exit(1)

target_workspace = workspaces[workspace_index]
target_workspace_id = target_workspace['id']
target_workspace_name = target_workspace['displayName']

print(f"🎯 Target workspace: {target_workspace_name}")
print(f"   ID: {target_workspace_id}")

# Option B: Or specify workspace ID directly
# target_workspace_id = "YOUR-WORKSPACE-GUID-HERE"

🎯 Target workspace: testautomatedcreation
   ID: c98a8692-0093-466a-b818-1fe71a28f3b7


In [5]:
# Step 1: List all items in the workspace
print(f"🔍 Checking items in workspace '{target_workspace_name}'")
print(f"   Workspace ID: {target_workspace_id}\n")

response = requests.get(
    f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}/items",
    headers=headers
)

if response.status_code == 200:
    all_items = response.json().get("value", [])
    
    # Filter out system items (SQLEndpoint, GraphModel are auto-generated)
    deletable_items = [
        item for item in all_items 
        if item['type'] not in ['SQLEndpoint', 'GraphModel']
    ]
    
    print(f"Found {len(deletable_items)} items that can be deleted:")
    for item in deletable_items:
        print(f"  • {item['type']}: {item['displayName']}")
    
    print(f"\n💡 Run the next cell to DELETE these items")
else:
    print(f"❌ Failed to list items: {response.status_code}")

🔍 Checking items in workspace 'testautomatedcreation'
   Workspace ID: c98a8692-0093-466a-b818-1fe71a28f3b7

Found 15 items that can be deleted:
  • Report: Vessels By Company
  • SemanticModel: maritimeSM
  • Lakehouse: maritimeLH
  • Eventhouse: maritimeEH
  • KQLDatabase: maritimeEH
  • Ontology: maritimeOntologyfromSM
  • Lakehouse: maritimeOntologyfromSM_lh_0849e3830d434eea93bc6a51626ab454
  • Notebook: runVesselswithSimulation2
  • Notebook: createOntology
  • Notebook: runVesselswithSimulation
  • Notebook: runVessels
  • Eventstream: maritimeES
  • DataAgent: maritimeDA
  • Reflex: RedAlertActivator
  • Map: vessels_map

💡 Run the next cell to DELETE these items


In [6]:
# Step 2: DELETE ALL ITEMS (handles dependencies with multiple passes)
import time

print(f"⚠️  DELETING ALL ITEMS from workspace '{target_workspace_name}'...\n")

# Define deletion order based on dependencies (reverse of deployment order)
priority_order = {
    'Report': 1,          # Delete reports first (depend on SemanticModel)
    'SemanticModel': 2,   # Then semantic models (depend on Lakehouse)
    'Reflex': 3,          # Then reflex (depends on Ontology)
    'DataAgent': 4,       # Then data agent
    'Map': 5,             # Then map
    'Notebook': 6,        # Then notebooks
    'Eventstream': 7,     # Then eventstream (depends on Eventhouse)
    'Ontology': 8,        # Then ontology (depends on SemanticModel)
    'KQLDatabase': 9,     # Then KQL DB (child of Eventhouse)
    'Eventhouse': 10,     # Then eventhouse
    'Lakehouse': 11       # Finally lakehouse
}

max_passes = 5  # Increased to 5 passes for stubborn dependencies
deleted_total = 0
retry_delay = 2  # Wait 2 seconds between passes to let Fabric settle

for pass_num in range(1, max_passes + 1):
    print(f"\n{'='*60}")
    print(f"🔄 Deletion Pass {pass_num}/{max_passes}")
    print(f"{'='*60}\n")
    
    response = requests.get(
        f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}/items",
        headers=headers
    )
    
    if response.status_code != 200:
        print(f"❌ Failed to list items: {response.status_code}")
        break
    
    all_items = response.json().get("value", [])
    
    # Filter out system items (SQLEndpoint, GraphModel are auto-generated)
    deletable_items = [
        item for item in all_items 
        if item['type'] not in ['SQLEndpoint', 'GraphModel']
    ]
    
    if not deletable_items:
        print("✅ No more items to delete!")
        break
    
    # Sort by priority order
    deletable_items.sort(key=lambda x: priority_order.get(x['type'], 99))
    
    print(f"📋 Found {len(deletable_items)} items to delete:")
    for item in deletable_items:
        print(f"   • {item['type']}: {item['displayName']}")
    
    print(f"\n🗑️  Deleting items...")
    deleted_count = 0
    failed_items = []
    
    for item in deletable_items:
        delete_url = f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}/items/{item['id']}"
        
        try:
            delete_response = requests.delete(delete_url, headers=headers)
            
            if delete_response.status_code in [200, 204]:
                print(f"   ✅ Deleted: {item['displayName']}")
                deleted_count += 1
                deleted_total += 1
            elif delete_response.status_code == 404:
                # Item already deleted (race condition)
                print(f"   ℹ️  Already gone: {item['displayName']}")
                deleted_count += 1
            else:
                error_msg = delete_response.text[:100] if delete_response.text else str(delete_response.status_code)
                print(f"   ⏳ Blocked: {item['displayName']} (status {delete_response.status_code})")
                failed_items.append(item)
        except Exception as e:
            print(f"   ❌ Error deleting {item['displayName']}: {str(e)[:50]}")
            failed_items.append(item)
    
    print(f"\n📊 Pass {pass_num} Summary:")
    print(f"   ✅ Deleted: {deleted_count}")
    print(f"   ⏳ Remaining: {len(failed_items)}")
    
    # If no progress, stop early
    if deleted_count == 0 and len(failed_items) > 0:
        print(f"\n⚠️  No progress made. Remaining items may have blocking dependencies:")
        for item in failed_items:
            print(f"      • {item['type']}: {item['displayName']}")
        
        if pass_num < max_passes:
            print(f"\n💡 You can:")
            print(f"   1. Run this cell again to retry")
            print(f"   2. Delete remaining items manually in Fabric portal")
        break
    
    # Small delay between passes to let Fabric catch up
    if pass_num < max_passes and len(failed_items) > 0:
        print(f"\n⏱️  Waiting {retry_delay} seconds before next pass...")
        time.sleep(retry_delay)

print(f"\n{'='*60}")
print(f"📊 FINAL SUMMARY")
print(f"{'='*60}")
print(f"✅ Total deleted: {deleted_total}")

# Check final state
response = requests.get(
    f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}/items",
    headers=headers
)

if response.status_code == 200:
    remaining = [
        item for item in response.json().get("value", [])
        if item['type'] not in ['SQLEndpoint', 'GraphModel']
    ]
    
    if remaining:
        print(f"⚠️  {len(remaining)} items still remain:")
        for item in remaining:
            print(f"   • {item['type']}: {item['displayName']}")
        print(f"\n💡 Options to proceed:")
        print(f"   1. Run this cell again - sometimes multiple attempts work")
        print(f"   2. Delete remaining items manually in Fabric portal")
        print(f"   3. Continue with deployment - some items may redeploy successfully")
    else:
        print(f"✅ Workspace is completely clean!")
        print(f"\n🚀 Ready for fresh deployment!")
        
        # CONDITIONAL COOLDOWN: Only wait if items were actually deleted
        if deleted_total > 0:
            print(f"\n⏱️  COOLDOWN: Waiting 20 seconds for Fabric to process deletions...")
            print(f"   (Prevents 'item not available yet' errors)")
            for remaining_seconds in range(20, 0, -5):
                print(f"   ⏳ {remaining_seconds} seconds remaining...")
                time.sleep(5)
            print(f"   ✅ Cooldown complete - workspace ready for deployment!")
        else:
            print(f"   ℹ️  No deletions performed - skipping cooldown")
            print(f"   🚀 Workspace ready for immediate deployment!")
else:
    print(f"❌ Failed to check final state: {response.status_code}")


⚠️  DELETING ALL ITEMS from workspace 'testautomatedcreation'...


🔄 Deletion Pass 1/5

📋 Found 15 items to delete:
   • Report: Vessels By Company
   • SemanticModel: maritimeSM
   • Reflex: RedAlertActivator
   • DataAgent: maritimeDA
   • Map: vessels_map
   • Notebook: runVesselswithSimulation2
   • Notebook: createOntology
   • Notebook: runVesselswithSimulation
   • Notebook: runVessels
   • Eventstream: maritimeES
   • Ontology: maritimeOntologyfromSM
   • KQLDatabase: maritimeEH
   • Eventhouse: maritimeEH
   • Lakehouse: maritimeLH
   • Lakehouse: maritimeOntologyfromSM_lh_0849e3830d434eea93bc6a51626ab454

🗑️  Deleting items...
   ✅ Deleted: Vessels By Company
   ✅ Deleted: maritimeSM
   ✅ Deleted: RedAlertActivator
   ✅ Deleted: maritimeDA
   ✅ Deleted: vessels_map
   ✅ Deleted: runVesselswithSimulation2
   ✅ Deleted: createOntology
   ✅ Deleted: runVesselswithSimulation
   ✅ Deleted: runVessels
   ✅ Deleted: maritimeES
   ✅ Deleted: maritimeOntologyfromSM
   ✅ Deleted: marit

In [7]:
# Enable required feature flags for selective deployment
from fabric_cicd import append_feature_flag

# Enable experimental features for items_to_include
append_feature_flag("enable_experimental_features")
append_feature_flag("enable_items_to_include")

print("✅ Feature flags enabled for phased deployment")

✅ Feature flags enabled for phased deployment


In [13]:
from fabric_cicd import FabricWorkspace, publish_all_items
import os
import time
import requests
import re

# 5. Deploy all items with dependency-aware phasing
print(f"\n🚀 Starting phased deployment to workspace '{target_workspace_name}'...")
print(f"   Source: {repo_path}")

# Helper function to verify items are deployed
def verify_items_deployed(expected_names, item_type, max_retries=5):
    """Verify that items are actually deployed and visible in the workspace"""
    for attempt in range(1, max_retries + 1):
        response = requests.get(
            f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}/items",
            headers=headers
        )
        
        if response.status_code == 200:
            items = response.json().get("value", [])
            deployed_names = [i['displayName'] for i in items if i['type'] == item_type]
            
            # Check if all expected items are present
            missing = [name for name in expected_names if name not in deployed_names]
            
            if not missing:
                print(f"   ✅ Verified: All {item_type} items are deployed")
                return True
            else:
                if attempt < max_retries:
                    print(f"   ⏳ Attempt {attempt}/{max_retries}: Waiting for {missing} to appear...")
                    time.sleep(5)
                else:
                    print(f"   ⚠️  Warning: Items not showing up: {missing}")
                    return False
        else:
            print(f"   ⚠️  API check failed: {response.status_code}")
            return False
    
    return False

# Helper function to upload files to Lakehouse Files section
def upload_to_lakehouse(lakehouse_id, local_file_path, target_path):
    """Upload a file to the Lakehouse Files section using OneLake API"""
    try:
        # Get a storage token for OneLake operations
        storage_token = credential.get_token("https://storage.azure.com/.default").token
        
        # Read the file content
        with open(local_file_path, 'rb') as f:
            file_content = f.read()
        
        # OneLake path: /workspaces/{workspaceId}/items/{lakehouseId}/Files/{path}
        onelake_url = f"https://onelake.dfs.fabric.microsoft.com/{target_workspace_id}/{lakehouse_id}/Files/{target_path}"
        
        # Use Azure Storage Data Lake Gen2 REST API
        upload_headers = {
            "Authorization": f"Bearer {storage_token}",
            "x-ms-version": "2021-06-08",
            "Content-Type": "application/octet-stream"
        }
        
        # Create the file (PUT with resource=file)
        create_response = requests.put(
            f"{onelake_url}?resource=file",
            headers=upload_headers
        )
        
        if create_response.status_code not in [201, 409]:  # 409 = file already exists, which is ok
            print(f"   ⚠️  Failed to create file: {create_response.status_code} - {create_response.text}")
            return False
        
        # Upload the file content (PATCH to append data)
        patch_headers = upload_headers.copy()
        patch_headers["Content-Length"] = str(len(file_content))
        
        patch_response = requests.patch(
            f"{onelake_url}?action=append&position=0",
            headers=patch_headers,
            data=file_content
        )
        
        if patch_response.status_code != 202:
            print(f"   ⚠️  Failed to append data: {patch_response.status_code} - {patch_response.text}")
            return False
        
        # Flush the data (PATCH with action=flush)
        flush_headers = upload_headers.copy()
        flush_headers["Content-Length"] = "0"
        
        flush_response = requests.patch(
            f"{onelake_url}?action=flush&position={len(file_content)}",
            headers=flush_headers
        )
        
        if flush_response.status_code not in [200, 201]:
            print(f"   ⚠️  Failed to flush file: {flush_response.status_code} - {flush_response.text}")
            return False
        
        print(f"   ✅ Uploaded: {target_path}")
        return True
        
    except Exception as e:
        print(f"   ❌ Error uploading {target_path}: {str(e)}")
        return False

# Initialize the Workspace configuration object with ALL item types
target_workspace_obj = FabricWorkspace(
    workspace_id=target_workspace_id,
    repository_directory=repo_path,
    token_credential=credential,
    # IMPORTANT: Include all item types present in the repo
    item_type_in_scope=[
        "Lakehouse",
        "Eventhouse",
        "KQLDatabase",
        "Notebook",
        "SemanticModel",
        "Report",
        "Reflex",
        "Eventstream",
        "DataAgent",
        "Ontology",
        "Map"
    ]
)

try:
    # Phase 1: Deploy data foundation (Lakehouse, Eventhouse + KQL Database)
    print("\n📋 Phase 1: Deploying data foundation (Lakehouse, Eventhouse + KQL Database)...")
    print("   ℹ️  Deploying ONLY Lakehouse and Eventhouse (with children)")
    
    # Use exclusion regex to exclude everything EXCEPT maritimeLH and maritimeEH
    # Pattern: exclude anything that's NOT maritimeLH or maritimeEH
    publish_all_items(
        target_workspace_obj,
        item_name_exclude_regex=r"^(?!maritimeLH$|maritimeEH$).*"
    )
    print("✅ Data foundation deployment completed")
    
    # VERIFY: Check that Lakehouse and Eventhouse actually deployed
    print("\n🔍 Verifying Phase 1 deployment...")
    verify_items_deployed(["maritimeLH"], "Lakehouse")
    verify_items_deployed(["maritimeEH"], "Eventhouse")
    
    # Brief wait for foundation to settle (verification already waited up to 25s)
    print("⏱️  Waiting 5 seconds...")
    time.sleep(5)
    
    # Upload GeoJSON files to Lakehouse Files section
    print("\n📁 Uploading GeoJSON files to Lakehouse...")
    
    # Get the deployed lakehouse ID for file upload
    response = requests.get(
        f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}/items",
        headers=headers
    )
    
    if response.status_code == 200:
        items = response.json().get("value", [])
        lakehouse = next((i for i in items if i['type'] == 'Lakehouse' and i['displayName'] == 'maritimeLH'), None)
        
        if lakehouse:
            lakehouse_id = lakehouse['id']
            
            # Upload the GeoJSON files from resources folder
            resources_folder = os.path.join(repo_path, "resources")
            geojson_files = [
                "HormuzShippingCorridor.geojson",
                "hormuz_risk_zone.geojson"
            ]
            
            for geojson_file in geojson_files:
                local_path = os.path.join(resources_folder, geojson_file)
                if os.path.exists(local_path):
                    upload_to_lakehouse(lakehouse_id, local_path, geojson_file)
                else:
                    print(f"   ⚠️  File not found: {geojson_file}")
        else:
            print("   ⚠️  Could not find lakehouse for file upload")
    else:
        print(f"   ❌ Failed to get lakehouse info: {response.status_code}")
    
    # PRE-PHASE 2: Update SemanticModel to reference the newly deployed lakehouse
    print("\n🔧 Pre-Phase 2: Updating SemanticModel lakehouse reference...")
    
    # Get the deployed lakehouse ID
    response = requests.get(
        f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}/items",
        headers=headers
    )
    
    if response.status_code == 200:
        items = response.json().get("value", [])
        lakehouse = next((i for i in items if i['type'] == 'Lakehouse' and i['displayName'] == 'maritimeLH'), None)
        
        if lakehouse:
            lakehouse_id = lakehouse['id']
            print(f"   📍 Found lakehouse: {lakehouse_id}")
            
            # Update expressions.tmdl with correct workspace and lakehouse IDs
            expressions_file = f"{repo_path}/maritimeSM.SemanticModel/definition/expressions.tmdl"
            
            if not os.path.exists(expressions_file):
                raise Exception(f"❌ expressions.tmdl not found at: {expressions_file}")
            
            with open(expressions_file, 'r') as f:
                content = f.read()
            
            # Replace old workspace/lakehouse IDs with new ones
            pattern = r'https://onelake\.dfs\.fabric\.microsoft\.com/[a-f0-9\-]+/[a-f0-9\-]+'
            new_url = f"https://onelake.dfs.fabric.microsoft.com/{target_workspace_id}/{lakehouse_id}"
            updated_content = re.sub(pattern, new_url, content)
            
            # Verify replacement happened
            if updated_content == content:
                print(f"   ⚠️  Warning: File content unchanged - pattern may not have matched")
            else:
                print(f"   ✅ Updated DirectLake connection to new lakehouse")
            
            with open(expressions_file, 'w') as f:
                f.write(updated_content)
            
        else:
            print("   ⚠️  Warning: Could not find deployed lakehouse - SemanticModel may fail")
    else:
        print(f"   ❌ Failed to fetch lakehouse info: {response.status_code}")
    
    # Phase 2: Deploy SemanticModel + Report (depends on Lakehouse)
    print("\n📋 Phase 2: Deploying SemanticModel + Report...")
    
    try:
        # Deploy SemanticModel and Report together (Report depends on SemanticModel)
        # Exclude: foundation items, notebooks, eventstream, dataagent, map, ontology, reflex
        publish_all_items(
            target_workspace_obj,
            item_name_exclude_regex=r"^(maritimeLH|maritimeEH|maritimeOntologyfromSM|RedAlertActivator|createOntology|runVessels|runVesselswithSimulation|runVesselswithSimulation2|maritimeES|maritimeDA|vessels_map)$"
        )
        print("✅ SemanticModel + Report deployment completed")
    except Exception as sm_error:
        print(f"\n❌ Phase 2 deployment FAILED:")
        print(f"   Error: {str(sm_error)}")
        import traceback
        traceback.print_exc()
        raise sm_error
    
    # VERIFY: Check that SemanticModel is deployed and ready
    print("\n🔍 Verifying Phase 2 deployment...")
    if not verify_items_deployed(["maritimeSM"], "SemanticModel", max_retries=8):
        raise Exception("SemanticModel deployment verification failed - cannot proceed to Ontology")
    
    # Wait for SemanticModel metadata processing (verification already waited up to 40s)
    print("⏱️  Waiting 10 seconds for metadata processing...")
    time.sleep(10)
    
    # Phase 3: Deploy Ontology (depends on SemanticModel)
    print("\n📋 Phase 3: Deploying Ontology...")
    try:
        # Exclude everything EXCEPT Ontology
        publish_all_items(
            target_workspace_obj,
            item_name_exclude_regex=r"^(?!maritimeOntologyfromSM$).*"
        )
        print("✅ Ontology deployment completed")
    except Exception as ontology_error:
        print(f"\n⚠️  Ontology deployment failed on first attempt: {ontology_error}")
        print("   Retrying after additional wait...")
        time.sleep(15)
        
        # Retry once
        publish_all_items(
            target_workspace_obj,
            item_name_exclude_regex=r"^(?!maritimeOntologyfromSM$).*"
        )
        print("✅ Ontology deployed on retry")
    
    # Verify Ontology
    print("\n🔍 Verifying Phase 3 deployment...")
    verify_items_deployed(["maritimeOntologyfromSM"], "Ontology")
    
    # Phase 4: Deploy Notebooks, Eventstream, DataAgent
    print("\n📋 Phase 4: Deploying Notebooks, Eventstream, DataAgent...")
    # Exclude already-deployed items and Map/Reflex (deploy those last)
    publish_all_items(
        target_workspace_obj,
        item_name_exclude_regex=r"^(maritimeLH|maritimeEH|maritimeSM|Vessels By Company|maritimeOntologyfromSM|RedAlertActivator|vessels_map)$"
    )
    print("✅ Supporting items deployment completed")
    
    # Phase 5: Deploy Reflex (depends on Ontology)
    print("\n📋 Phase 5: Deploying Reflex...")
    publish_all_items(
        target_workspace_obj,
        item_name_exclude_regex=r"^(?!RedAlertActivator$).*"
    )
    print("✅ Reflex deployment completed")
    
    # Phase 6: Deploy Map (depends on Ontology)
    print("\n📋 Phase 6: Deploying Map (depends on Ontology)...")
    try:
        # Check if vessels_map folder exists
        map_folder = f"{repo_path}/vessels_map.Map"
        if os.path.exists(map_folder):
            publish_all_items(
                target_workspace_obj,
                item_name_exclude_regex=r"^(?!vessels_map$).*"
            )
            print("✅ Map deployment completed")
        else:
            print("   ℹ️  Map folder not found - skipping")
    except Exception as map_error:
        print(f"\n⚠️  Map deployment failed: {map_error}")
        print("   💡 Map items may need manual creation in Fabric portal")
        # Don't raise - Map failure is non-critical, continue with verification
    
    # Final verification
    print("\n🔍 Final verification - checking all items...")
    response = requests.get(
        f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}/items",
        headers=headers
    )
    
    if response.status_code == 200:
        final_items = response.json().get("value", [])
        print(f"\n✅ Deployment complete! Total items in workspace: {len(final_items)}")
        
        by_type = {}
        for item in final_items:
            item_type = item['type']
            by_type[item_type] = by_type.get(item_type, 0) + 1
        
        print("\n📊 Items by type:")
        for item_type, count in sorted(by_type.items()):
            print(f"   {item_type}: {count}")
    
    print(f"\n✅ Maritime Demo deployed successfully! All items with dependencies resolved.")
    print(f"\n📊 View your workspace: https://app.fabric.microsoft.com/groups/{target_workspace_id}")
    
except Exception as e:
    print(f"\n❌ Deployment failed: {str(e)}")
    print(f"\n💡 Troubleshooting steps:")
    print(f"   1. Run Cell 10 to check what deployed successfully")
    print(f"   2. Check the Fabric portal to see if items are visible")
    print(f"   3. Run Cell 11 to retry missing items")
    print(f"   4. Check error details above for specific issues")
    raise


[info]   21:05:42 - 
[info]   21:05:42 - ####################################################################################################
[info]   21:05:42 - ########## Validating Parameter File ###############################################################
[info]   21:05:42 - ####################################################################################################
[info]   21:05:42 - 
[warn]   21:05:42 - Parameter file not found with path: /Users/rabindori/workspace/aitour2/maritimedemo/fabricdemo/parameter.yml
[warn]   21:05:42 - Validation terminated: not found



🚀 Starting phased deployment to workspace 'testautomatedcreation'...
   Source: /Users/rabindori/workspace/aitour2/maritimedemo/fabricdemo

📋 Phase 1: Deploying data foundation (Lakehouse, Eventhouse + KQL Database)...
   ℹ️  Deploying ONLY Lakehouse and Eventhouse (with children)


[info]   21:05:46 - 
[info]   21:05:46 - ####################################################################################################
[info]   21:05:46 - ########## Publishing Workspace Folders ############################################################
[info]   21:05:46 - ####################################################################################################
[info]   21:05:46 - 
[info]   21:05:46 - Publishing Workspace Folders
         21:05:46 - Published
[warn]   21:05:48 - Using item_name_exclude_regex is risky as it can prevent needed dependencies from being deployed.  Use at your own risk.
[info]   21:05:48 - 
[info]   21:05:48 - ####################################################################################################
[info]   21:05:48 - ########## Publishing Item 4/29: Lakehouse #########################################################
[info]   21:05:48 - ############################################################################################

✅ Data foundation deployment completed

🔍 Verifying Phase 1 deployment...
   ✅ Verified: All Lakehouse items are deployed
   ✅ Verified: All Eventhouse items are deployed
⏱️  Waiting 5 seconds...

📁 Uploading GeoJSON files to Lakehouse...
   ✅ Uploaded: HormuzShippingCorridor.geojson
   ✅ Uploaded: hormuz_risk_zone.geojson

🔧 Pre-Phase 2: Updating SemanticModel lakehouse reference...
   📍 Found lakehouse: 0410d25b-58f7-4c01-83c5-182fdebc3068
   ⚠️  Warning: File content unchanged - pattern may not have matched

📋 Phase 2: Deploying SemanticModel + Report...


[info]   21:06:50 - 
[info]   21:06:50 - ####################################################################################################
[info]   21:06:50 - ########## Publishing Workspace Folders ############################################################
[info]   21:06:50 - ####################################################################################################
[info]   21:06:50 - 
[info]   21:06:50 - Publishing Workspace Folders
         21:06:50 - Published
[warn]   21:06:52 - Using item_name_exclude_regex is risky as it can prevent needed dependencies from being deployed.  Use at your own risk.
[info]   21:06:52 - 
[info]   21:06:52 - ####################################################################################################
[info]   21:06:52 - ########## Publishing Item 4/29: Lakehouse #########################################################
[info]   21:06:52 - ############################################################################################

...

✅ Supporting items deployment completed

📋 Phase 5: Deploying Reflex...


[info]   21:08:04 - 
[info]   21:08:04 - ####################################################################################################
[info]   21:08:04 - ########## Publishing Workspace Folders ############################################################
[info]   21:08:04 - ####################################################################################################
[info]   21:08:04 - 
[info]   21:08:04 - Publishing Workspace Folders
         21:08:04 - Published
[warn]   21:08:05 - Using item_name_exclude_regex is risky as it can prevent needed dependencies from being deployed.  Use at your own risk.
[info]   21:08:05 - 
[info]   21:08:05 - ####################################################################################################
[info]   21:08:05 - ########## Publishing Item 4/29: Lakehouse #########################################################
[info]   21:08:05 - ############################################################################################

✅ Reflex deployment completed

📋 Phase 6: Deploying Map (depends on Ontology)...


[info]   21:08:17 - 
[info]   21:08:17 - ####################################################################################################
[info]   21:08:17 - ########## Publishing Workspace Folders ############################################################
[info]   21:08:17 - ####################################################################################################
[info]   21:08:17 - 
[info]   21:08:17 - Publishing Workspace Folders
         21:08:17 - Published
[warn]   21:08:19 - Using item_name_exclude_regex is risky as it can prevent needed dependencies from being deployed.  Use at your own risk.
[info]   21:08:19 - 
[info]   21:08:19 - ####################################################################################################
[info]   21:08:19 - ########## Publishing Item 4/29: Lakehouse #########################################################
[info]   21:08:19 - ############################################################################################

✅ Map deployment completed

🔍 Final verification - checking all items...

✅ Deployment complete! Total items in workspace: 18

📊 Items by type:
   DataAgent: 1
   Eventhouse: 1
   Eventstream: 1
   GraphModel: 1
   KQLDatabase: 1
   Lakehouse: 2
   Map: 1
   Notebook: 4
   Ontology: 1
   Reflex: 1
   Report: 1
   SQLEndpoint: 2
   SemanticModel: 1

✅ Maritime Demo deployed successfully! All items with dependencies resolved.

📊 View your workspace: https://app.fabric.microsoft.com/groups/c98a8692-0093-466a-b818-1fe71a28f3b7


In [ ]:
# Update Ontology Resource Links to point to the deployed "Vessels By Company" report
print("\n🔗 Updating Ontology resource links to point to Vessels By Company report...")

# Get the deployed report ID
response = requests.get(
    f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}/items",
    headers=headers
)

if response.status_code == 200:
    items = response.json().get("value", [])
    report = next((i for i in items if i['type'] == 'Report' and i['displayName'] == 'Vessels By Company'), None)
    
    if report:
        report_id = report['id']
        print(f"   📊 Found report: {report_id}")
        
        # Update the ResourceLinks definition
        resource_links_file = f"{repo_path}/maritimeOntologyfromSM.Ontology/EntityTypes/98576905401063/ResourceLinks/definition.json"
        
        if os.path.exists(resource_links_file):
            # Create the updated resource links
            updated_links = {
                "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/ontology/resourceLinks/1.0.0/schema.json",
                "resourceLinks": [
                    {
                        "type": "PowerBIReport",
                        "workspaceId": target_workspace_id,
                        "itemId": report_id
                    }
                ]
            }
            
            with open(resource_links_file, 'w') as f:
                json.dump(updated_links, f, indent=2)
            
            print(f"   ✅ Updated resource links to point to Vessels By Company report")
            
            # Redeploy the Ontology to apply the change
            print(f"\n🚀 Redeploying Ontology with updated resource links...")
            
            publish_all_items(
                target_workspace_obj,
                item_name_exclude_regex=r"^(?!maritimeOntologyfromSM$).*"
            )
            
            print(f"✅ Ontology redeployed with correct report link!")
        else:
            print(f"   ⚠️  ResourceLinks file not found: {resource_links_file}")
    else:
        print("   ⚠️  Vessels By Company report not found in workspace")
else:
    print(f"   ❌ Failed to get workspace items: {response.status_code}")

In [11]:
# ⚠️ RUN THIS CELL AFTER DEPLOYMENT (Cell 9) TO SEE ACTUAL RESULTS
# Check what's actually deployed vs what's in Git repo
import os

print("🔍 Comparing Git repository vs deployed items...")
print("⚠️  This fetches FRESH data from Fabric workspace\n")

# 1. Scan Git repository for Fabric items
print("📂 Items in Git repository:")
git_items = {}
for entry in os.listdir(repo_path):
    if os.path.isdir(os.path.join(repo_path, entry)) and '.' in entry:
        parts = entry.split('.')
        if len(parts) == 2:
            item_name, item_type = parts
            if item_type not in git_items:
                git_items[item_type] = []
            git_items[item_type].append(item_name)

for item_type, names in sorted(git_items.items()):
    print(f"  📦 {item_type} ({len(names)}): {', '.join(names)}")

# 2. Fetch FRESH deployed items from workspace
print(f"\n📊 Items deployed to workspace (FRESH FETCH):")
response = requests.get(
    f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}/items",
    headers=headers
)

if response.status_code == 200:
    # Store fresh items for use by next cells
    items = response.json().get("value", [])
    globals()['items'] = items  # Update global variable with fresh data
    
    # Group by type
    deployed_items = {}
    for item in items:
        item_type = item.get('type', 'Unknown')
        if item_type not in deployed_items:
            deployed_items[item_type] = []
        deployed_items[item_type].append(item['displayName'])
    
    # Store for next cell
    globals()['deployed_items'] = deployed_items
    
    for item_type, names in sorted(deployed_items.items()):
        print(f"  📦 {item_type} ({len(names)}): {', '.join(names)}")
    
    # 3. Compare and show what's missing
    print(f"\n❓ Missing items (in Git but not deployed):")
    
    # Map Git folder extensions to Fabric item types
    type_mapping = {
        'Reflex': 'Reflex',
        'DataAgent': 'DataAgent', 
        'Map': 'Map',
        'Eventstream': 'Eventstream',
        'Ontology': 'Ontology',
        'Eventhouse': 'Eventhouse',
        'KQLDatabase': 'KQLDatabase',
        'Lakehouse': 'Lakehouse',
        'Notebook': 'Notebook',
        'SemanticModel': 'SemanticModel',
        'Report': 'Report'
    }
    
    missing_count = 0
    for git_type, git_names in git_items.items():
        fabric_type = type_mapping.get(git_type, git_type)
        deployed_names = deployed_items.get(fabric_type, [])
        
        for git_name in git_names:
            if git_name not in deployed_names:
                print(f"  ⚠️ {git_type}: {git_name}")
                missing_count += 1
    
    if missing_count == 0:
        print(f"  ✅ All items from Git are deployed!")
    
    # 4. Special check for KQL Database (child of Eventhouse)
    print(f"\n🔍 KQL Database check:")
    if 'KQLDatabase' in deployed_items:
        print(f"  ✅ KQL Database deployed: {', '.join(deployed_items['KQLDatabase'])}")
    elif 'Eventhouse' in deployed_items:
        print(f"  ⚠️  Eventhouse exists but KQL Database not showing up yet")
        print(f"     💡 KQL Database may still be initializing - wait a minute and re-run this cell")
    else:
        print(f"  ❌ No Eventhouse or KQL Database found")
    
    print(f"\n📊 Summary:")
    print(f"  Git items: {sum(len(v) for v in git_items.values())} across {len(git_items)} types")
    print(f"  Deployed: {len(items)} items across {len(deployed_items)} types")
    print(f"  Missing: {missing_count}")
    
    if missing_count == 0:
        print(f"\n🎉 SUCCESS! All items deployed correctly!")
    else:
        print(f"\n⚠️  Run cell 11 to retry deploying missing items")
    
else:
    print(f"❌ Failed to list deployed items: {response.status_code}")

🔍 Comparing Git repository vs deployed items...
⚠️  This fetches FRESH data from Fabric workspace

📂 Items in Git repository:
  📦 DataAgent (1): maritimeDA
  📦 Eventhouse (1): maritimeEH
  📦 Eventstream (1): maritimeES
  📦 Lakehouse (1): maritimeLH
  📦 Map (1): vessels_map
  📦 Notebook (4): createOntology, runVesselswithSimulation2, runVesselswithSimulation, runVessels
  📦 Ontology (1): maritimeOntologyfromSM
  📦 Reflex (1): RedAlertActivator
  📦 Report (1): Vessels By Company
  📦 SemanticModel (1): maritimeSM
  📦 git (1): 
  📦 vscode (1): 

📊 Items deployed to workspace (FRESH FETCH):
  📦 DataAgent (1): maritimeDA
  📦 Eventhouse (1): maritimeEH
  📦 Eventstream (1): maritimeES
  📦 GraphModel (1): maritimeOntologyfromSM_graph_a67f4ae98b374446b8dff5b1fff8f8ad
  📦 KQLDatabase (1): maritimeEH
  📦 Lakehouse (2): maritimeLH, maritimeOntologyfromSM_lh_a67f4ae98b374446b8dff5b1fff8f8ad
  📦 Map (1): vessels_map
  📦 Notebook (4): runVesselswithSimulation2, runVessels, runVesselswithSimulation, cr

In [ ]:
# Deploy missing items explicitly (if any were skipped)
print("\n🔧 Checking for and deploying any missing items...\n")

# Items that are commonly skipped or have issues
problematic_items = [
    ("RedAlertActivator.Reflex", "Reflex"),
    ("maritimeES.Eventstream", "Eventstream"),
    ("maritimeDA.DataAgent", "DataAgent"),
    ("vessels_map.Map", "Map"),
    ("maritimeEH.KQLDatabase", "KQLDatabase")
]

items_to_retry = []

# Check which ones are actually missing
for item_folder, item_type in problematic_items:
    item_name = item_folder.split('.')[0]
    
    # Check if folder exists in Git
    item_path = os.path.join(repo_path, item_folder)
    if not os.path.exists(item_path):
        print(f"⏭️  Skipping {item_name} - folder not found in Git")
        continue
    
    # Check if already deployed
    if item_type in deployed_items and item_name in deployed_items[item_type]:
        print(f"✅ {item_name} already deployed")
        continue
    
    print(f"⚠️  {item_name} is missing - will retry deployment")
    items_to_retry.append(item_folder)

if items_to_retry:
    print(f"\n🚀 Deploying {len(items_to_retry)} missing items...")
    print(f"   Items: {', '.join(items_to_retry)}\n")
    
    try:
        publish_all_items(
            target_workspace_obj,
            items_to_include=items_to_retry
        )
        print(f"\n✅ Successfully deployed missing items!")
        
        # Verify
        print(f"\n🔍 Verifying deployment...")
        response = requests.get(
            f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}/items",
            headers=headers
        )
        
        if response.status_code == 200:
            items = response.json().get("value", [])
            print(f"✅ Total items now: {len(items)}")
        
    except Exception as e:
        print(f"\n❌ Failed to deploy some items: {e}")
        print(f"\n💡 This may be due to:")
        print(f"   1. Item type not fully supported by fabric-cicd")
        print(f"   2. Missing dependencies")
        print(f"   3. Items need manual creation in Fabric portal")
else:
    print(f"\n✅ No missing items to deploy!")

In [ ]:
# Update createOntology notebook to use dynamic Kusto cluster parameter
print("\n🔧 Parameterizing createOntology notebook with dynamic Kusto cluster...\n")

import re

# Read the current notebook
notebook_path = f"{repo_path}/createOntology.Notebook/notebook-content.py"
with open(notebook_path, 'r') as f:
    content = f.read()

# Find the hardcoded kusto_cluster line
old_line_pattern = r'kusto_cluster = "https://[^"]+\.kusto\.fabric\.microsoft\.com"'

# Check if parameters cell already exists
if '# PARAMETERS CELL' not in content:
    # Add a parameters cell at the beginning (after metadata)
    # Find the end of the first metadata block
    first_cell_end = content.find('# CELL ********************')
    
    if first_cell_end > 0:
        # Insert parameter cell before the first code cell
        parameter_cell = '''# CELL ********************

# PARAMETERS CELL
# These can be overridden when running the notebook via API or mssparkutils.notebook.run()
kusto_cluster = "https://placeholder.kusto.fabric.microsoft.com"  # Will be set dynamically
kusto_db = "maritimeEH"

# METADATA ********************

# META {
# META   "language": "python",
# META   "language_group": "synapse_pyspark"
# META }

'''
        updated_content = content[:first_cell_end] + parameter_cell + content[first_cell_end:]
        
        # Now remove the hardcoded kusto_cluster assignment in the later cell
        # Keep kusto_db assignment but remove kusto_cluster
        updated_content = re.sub(
            r'^kusto_cluster = "https://[^"]+\.kusto\.fabric\.microsoft\.com"\s*\n',
            '',
            updated_content,
            flags=re.MULTILINE
        )
        
        # Also remove duplicate kusto_db if it exists later
        lines = updated_content.split('\n')
        seen_kusto_db = False
        filtered_lines = []
        for line in lines:
            if line.strip().startswith('kusto_db = "maritimeEH"'):
                if not seen_kusto_db:
                    filtered_lines.append(line)
                    seen_kusto_db = True
                # Skip duplicate
            else:
                filtered_lines.append(line)
        updated_content = '\n'.join(filtered_lines)
        
        # Write back
        with open(notebook_path, 'w') as f:
            f.write(updated_content)
        
        print("✅ Added parameter cell to createOntology notebook")
        print("   Parameter: kusto_cluster (default: placeholder)")
        print("\n📝 Updated local notebook file")
        
        # Now redeploy the notebook
        print("\n🚀 Redeploying createOntology notebook...")
        publish_all_items(
            target_workspace_obj,
            items_to_include=["createOntology.Notebook"]
        )
        print("✅ Notebook redeployed with parameterization")
        
        print("\n💡 To run this notebook with the correct Kusto cluster:")
        print("   1. Get the Eventhouse query URI from the Fabric portal")
        print("   2. Or use: mssparkutils.notebook.run('createOntology', parameters={'kusto_cluster': 'YOUR_CLUSTER_URL'})")
    else:
        print("⚠️ Could not find cell markers in notebook")
else:
    print("✅ Notebook already has a parameters cell")
    print("   Skipping parameterization")

In [ ]:
# Get the Eventhouse Kusto cluster URI and update createOntology notebook
print("\n🔍 Retrieving Eventhouse Kusto cluster URI...\n")

eventhouse = next((i for i in items if i['type'] == 'Eventhouse' and i['displayName'] == 'maritimeEH'), None)
kql_db = next((i for i in items if i['type'] == 'KQLDatabase' and i['displayName'] == 'maritimeEH'), None)

if eventhouse and kql_db:
    eventhouse_id = eventhouse['id']
    kql_db_id = kql_db['id']
    
    # Get KQL Database properties which include the query URI
    db_props_url = f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id}/kqlDatabases/{kql_db_id}"
    db_response = requests.get(db_props_url, headers=headers)
    
    if db_response.status_code == 200:
        db_data = db_response.json()
        properties = db_data.get('properties', {})
        
        # The queryServiceUri is the Kusto cluster endpoint
        kusto_uri = properties.get('queryServiceUri')
        
        if kusto_uri:
            print(f"✅ Retrieved Kusto cluster URI: {kusto_uri}")
            
            # Now update the createOntology notebook with this URI as the default
            notebook_path = f"{repo_path}/createOntology.Notebook/notebook-content.py"
            
            with open(notebook_path, 'r') as f:
                content = f.read()
            
            # Replace the placeholder with the actual cluster URI
            updated_content = content.replace(
                'kusto_cluster = "https://placeholder.kusto.fabric.microsoft.com"',
                f'kusto_cluster = "{kusto_uri}"'
            )
            
            with open(notebook_path, 'w') as f:
                f.write(updated_content)
            
            print(f"✅ Updated notebook parameter with real cluster URI")
            
            # Redeploy the notebook with the correct URI
            print(f"\n🚀 Redeploying createOntology notebook with Kusto cluster URI...")
            publish_all_items(
                target_workspace_obj,
                items_to_include=["createOntology.Notebook"]
            )
            print(f"✅ Notebook redeployed and ready to run!")
            
            print(f"\n📋 The notebook is now configured with:")
            print(f"   Kusto Cluster: {kusto_uri}")
            print(f"   Database: maritimeEH")
            print(f"\n💡 You can now run the notebook directly from Fabric portal without configuration!")
        else:
            print(f"⚠️ queryServiceUri not found in KQL Database properties")
            print(f"   Properties: {properties.keys()}")
    else:
        print(f"❌ Failed to get KQL Database properties: {db_response.status_code}")
        print(f"   Response: {db_response.text}")
else:
    print("⚠️ Eventhouse or KQL Database not found")